# Lab 7: Text Classification with Stack Overflow Data

**Dataset:** `bigquery-public-data.stackoverflow.posts_questions`
**Task:** Predict which tag (python, javascript, java, c#, html) a question belongs to
**Pipeline:** BigQuery → GCS → Local baseline → Vertex AI training → Batch prediction → Metadata

**Key exam topics:** Text preprocessing, NLP pipelines, TF/Keras custom training,
batch vs. online serving, pre-built containers, model registration

---

## Part 0: Setup

In [33]:
import os
from google.cloud import bigquery, aiplatform

PROJECT_ID = "carty-470812"
REGION = "us-central1"
BUCKET_NAME = f"{PROJECT_ID}-lab7-stackoverflow"
BUCKET_URI = f"gs://{BUCKET_NAME}"

# Google's pre-built containers
# Full list: https://cloud.google.com/vertex-ai/docs/training/pre-built-containers
TRAIN_IMAGE = "us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-15.py310:latest"
SERVE_IMAGE = "us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest"

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)
bq = bigquery.Client(project=PROJECT_ID)

# Create bucket (idempotent)
!gcloud storage buckets create {BUCKET_URI} --location={REGION} --project={PROJECT_ID} 2>/dev/null || echo "Bucket already exists"

print(f"Bucket:      {BUCKET_URI}")
print(f"Train image: {TRAIN_IMAGE}")
print(f"Serve image: {SERVE_IMAGE}")

Bucket already exists
Bucket:      gs://carty-470812-lab7-stackoverflow
Train image: us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-15.py310:latest
Serve image: us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest


## Part 1: Data Exploration & Preparation in BigQuery

Explore the Stack Overflow public dataset, sample 20k questions per tag,
clean HTML from body text, and create reproducible train/val/test splits.

### 1a. Explore the raw data

In [34]:
# What does the data look like?
query = """
SELECT id, title, body, tags, score, creation_date
FROM `bigquery-public-data.stackoverflow.posts_questions`
LIMIT 5
"""
bq.query(query).to_dataframe()

/Users/james.carty/Documents/VScode/google-ml-engineer/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,title,body,tags,score,creation_date
0,24961052,Use scroll bars on child controls in wpf,<p>I have an explorer type window that I am tr...,c#|wpf,1,2014-07-25 17:13:59.017000+00:00
1,24997648,LinqDataSource Result becoming Null before OnS...,<p>I'm assigning list to e.Result in LinqData...,c#|linqdatasource,1,2014-07-28 14:35:46.723000+00:00
2,25003102,Make CaptionText Bold in CaptionPanel gwt,<p>I want to make the captionText in captionPa...,java|gwt,0,2014-07-28 19:45:22.933000+00:00
3,25009668,Strange behaviour with Type.createInstance in ...,<p>Shouldn't the following code throw an error...,reflection|types|haxe,0,2014-07-29 06:52:02.930000+00:00
4,25030415,angularjs filter to convert to valid directive...,<p>I want to write filter that turns for examp...,angularjs|filter|angularjs-directive|angularjs...,1,2014-07-30 06:51:13.953000+00:00


In [35]:
# Count questions per tag for our 5 target tags
query = """
SELECT
  tag,
  COUNT(*) AS question_count
FROM
  `bigquery-public-data.stackoverflow.posts_questions`,
  UNNEST(SPLIT(tags, '|')) AS tag
WHERE tag IN ('python', 'javascript', 'java', 'c#', 'html')
GROUP BY tag
ORDER BY question_count DESC
"""
bq.query(query).to_dataframe()

/Users/james.carty/Documents/VScode/google-ml-engineer/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,tag,question_count
0,javascript,2426570
1,python,2026601
2,java,1866055
3,c#,1559381
4,html,1146211


### 1b. Create the cleaned, sampled, split dataset

Strategy:
- 20,000 questions per tag (100k total) for balanced classes
- Strip HTML tags from body, concatenate title + body as `text`
- Hash-based split (FARM_FINGERPRINT) for reproducibility: 80/10/10

In [36]:
# Create BQ dataset in US multi-region (required to join with public data)
dataset = bigquery.Dataset(f"{PROJECT_ID}.lab7_stackoverflow")
dataset.location = "US"
bq.create_dataset(dataset, exists_ok=True)
print("Dataset created in US!")

Dataset created in US!


In [38]:
# Create the cleaned, sampled, split table
query = """
CREATE OR REPLACE TABLE `carty-470812.lab7_stackoverflow.questions_split` AS

WITH filtered AS (
  SELECT
    id,
    title,
    REGEXP_REPLACE(body, r'<[^>]+>', ' ') AS body_clean,  -- strip HTML tags
    tag,
    ROW_NUMBER() OVER (PARTITION BY tag ORDER BY FARM_FINGERPRINT(CAST(id AS STRING))) AS rn
  FROM
    `bigquery-public-data.stackoverflow.posts_questions`,
    UNNEST(SPLIT(tags, '|')) AS tag
  WHERE tag IN ('python', 'javascript', 'java', 'c#', 'html')
),

sampled AS (
  SELECT * FROM filtered WHERE rn <= 20000
)

SELECT
  id,
  CONCAT(title, ' ', body_clean) AS text,
  tag AS label,
  CASE
    WHEN MOD(ABS(FARM_FINGERPRINT(CAST(id AS STRING))), 10) < 8 THEN 'train'
    WHEN MOD(ABS(FARM_FINGERPRINT(CAST(id AS STRING))), 10) = 8 THEN 'val'
    ELSE 'test'
  END AS split
FROM sampled
"""

bq.query(query).result()
print("Table created!")

Table created!


In [ ]:
# Verify split distribution — should be ~80/10/10 per label
query = """
SELECT split, label, COUNT(*) AS n
FROM `carty-470812.lab7_stackoverflow.questions_split`
GROUP BY split, label
ORDER BY split, label
"""
bq.query(query).to_dataframe()

/Users/james.carty/Documents/VScode/google-ml-engineer/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,split,label,n
0,test,c#,2081
1,test,html,1954
2,test,java,1968
3,test,javascript,1985
4,test,python,2055
5,train,c#,15914
6,train,html,15993
7,train,java,16024
8,train,javascript,16078
9,train,python,15913


### 1c. Export to GCS

In [ ]:
# Export each split to GCS as CSV
for split in ['train', 'val', 'test']:
    query = f"""
    SELECT text, label
    FROM `carty-470812.lab7_stackoverflow.questions_split`
    WHERE split = '{split}'
    """
    dest_uri = f"{BUCKET_URI}/data/{split}.csv"

    df = bq.query(query).to_dataframe()
    df.to_csv(f"/tmp/{split}.csv", index=False)
    !gcloud storage cp /tmp/{split}.csv {dest_uri}
    print(f"{split}: {len(df)} rows → {dest_uri}")

/Users/james.carty/Documents/VScode/google-ml-engineer/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Copying file:///tmp/train.csv to gs://carty-470812-lab7-stackoverflow/data/train.csv
  Completed files 1/1 | 130.2MiB/130.2MiB | 12.4MiB/s                          

Average throughput: 13.4MiB/s
train: 79922 rows → gs://carty-470812-lab7-stackoverflow/data/train.csv
Copying file:///tmp/val.csv to gs://carty-470812-lab7-stackoverflow/data/val.csv
  Completed files 1/1 | 16.1MiB/16.1MiB                                        

Average throughput: 55.0MiB/s
val: 10035 rows → gs://carty-470812-lab7-stackoverflow/data/val.csv
Copying file:///tmp/test.csv to gs://carty-470812-lab7-stackoverflow/data/test.csv
  Completed files 1/1 | 16.4MiB/16.4MiB                                        

Average throughput: 52.3MiB/s
test: 10043 rows → gs://carty-470812-lab7-stackoverflow/data/test.csv


## Part 2: Text Preprocessing & TF-IDF Baseline

Before building a neural model, establish a baseline with TF-IDF + Logistic Regression.
This is a critical pattern: always start with a simple model to set expectations.

In [ ]:
import pandas as pd

train_df = pd.read_csv("/tmp/train.csv")
val_df = pd.read_csv("/tmp/val.csv")
test_df = pd.read_csv("/tmp/test.csv")

print(f"Train: {train_df.shape}")
print(f"Val:   {val_df.shape}")
print(f"Test:  {test_df.shape}")

print(f"\nLabel distribution:\n{train_df['label'].value_counts()}")
print(f"\nText length stats:")
print(f"  Average: {train_df['text'].str.len().mean():.0f} chars")
print(f"  Median:  {train_df['text'].str.len().median():.0f} chars")
print(f"  Max:     {train_df['text'].str.len().max():.0f} chars")
print(f"  95th %%:  {train_df['text'].str.len().quantile(0.95):.0f} chars")

Train: (79922, 2)
Val:   (10035, 2)
Test:  (10043, 2)

Label distribution:
label
javascript    16078
java          16024
html          15993
c#            15914
python        15913
Name: count, dtype: int64

Text length stats:
  Average: 1684 chars
  Median:  1061 chars
  Max:     35459 chars
  95th %%:  5016 chars


### TF-IDF + Logistic Regression baseline

TF-IDF converts text to numerical features based on word frequency.
For topic classification where keywords matter most (language names, library names),
this is a strong baseline that often beats simple neural models.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import time

# TF-IDF + Logistic Regression
print("Fitting TF-IDF...")
t0 = time.time()
tfidf = TfidfVectorizer(max_features=10000, stop_words='english')
X_train = tfidf.fit_transform(train_df['text'])
X_val = tfidf.transform(val_df['text'])
print(f"TF-IDF done in {time.time()-t0:.1f}s — vocab size: {len(tfidf.vocabulary_)}")

print("\nTraining Logistic Regression...")
t0 = time.time()
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, train_df['label'])
print(f"Training done in {time.time()-t0:.1f}s")

val_preds = lr.predict(X_val)
print(f"\nValidation Accuracy: {accuracy_score(val_df['label'], val_preds):.4f}")
print(f"\n{classification_report(val_df['label'], val_preds)}")

Fitting TF-IDF...
TF-IDF done in 7.7s — vocab size: 10000

Training Logistic Regression...
Training done in 7.2s

Validation Accuracy: 0.8387

              precision    recall  f1-score   support

          c#       0.89      0.89      0.89      2005
        html       0.72      0.75      0.74      2053
        java       0.92      0.91      0.92      2008
  javascript       0.73      0.69      0.71      1937
      python       0.93      0.95      0.94      2032

    accuracy                           0.84     10035
   macro avg       0.84      0.84      0.84     10035
weighted avg       0.84      0.84      0.84     10035



**Observation:** 83.8% with TF-IDF + Logistic Regression in ~10 seconds.
Python and Java are easiest to classify (91-95% recall) while JavaScript and HTML
are hardest (69-75%) — makes sense since HTML/JS questions overlap heavily.

This is our number to beat.

## Part 3: Neural Model — Local Training

Build an Embedding + GlobalAveragePooling1D model with TensorFlow.
Train locally first to validate the architecture before submitting to Vertex AI.

**Architecture choice:** GlobalAveragePooling1D (not LSTM) because word ORDER doesn't
matter for topic classification — "python list sort" and "sort list python" should
classify identically. This is faster to train and generalizes better.

**Critical fix:** Truncate text to 500 chars. Stack Overflow posts can be 35k+ chars,
and TextVectorization must process the full string before truncating to max_length tokens.
Without truncation, training is 100x slower.

In [ ]:
import tensorflow as tf

# Truncate text — first 500 chars contains title + opening paragraph
train_df["text"] = train_df["text"].str[:500]
val_df["text"] = val_df["text"].str[:500]
test_df["text"] = test_df["text"].str[:500]

# Encode labels
labels = sorted(train_df["label"].unique())
label_to_id = {l: i for i, l in enumerate(labels)}
print(f"Labels: {label_to_id}")

y_train = train_df["label"].map(label_to_id).values
y_val = val_df["label"].map(label_to_id).values
y_test = test_df["label"].map(label_to_id).values

# TextVectorization — converts text to integer token sequences
# We run this ONCE as a preprocessing step, not inside the model,
# to avoid re-tokenizing 80k strings every epoch
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=10000, output_mode="int", output_sequence_length=128
)
vectorizer.adapt(train_df["text"].values)
print(f"Vocabulary size: {vectorizer.vocabulary_size()}")

# Pre-vectorize all splits
print("Pre-vectorizing...")
X_train = vectorizer(train_df["text"].values).numpy()
X_val = vectorizer(val_df["text"].values).numpy()
X_test = vectorizer(test_df["text"].values).numpy()
print(f"X_train shape: {X_train.shape}")

Labels: {'c#': 0, 'html': 1, 'java': 2, 'javascript': 3, 'python': 4}
Vocabulary size: 10000
Pre-vectorizing...
X_train shape: (79922, 128)


In [ ]:
# Build model — takes pre-vectorized integer sequences
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(10000, 64, input_length=128),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(labels), activation="softmax"),
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10, batch_size=64,
)

/Users/james.carty/Documents/VScode/google-ml-engineer/.venv/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6258 - loss: 0.9324 - val_accuracy: 0.7935 - val_loss: 0.5587
Epoch 2/10
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8000 - loss: 0.5441 - val_accuracy: 0.8010 - val_loss: 0.5209
Epoch 3/10
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8200 - loss: 0.4802 - val_accuracy: 0.8146 - val_loss: 0.4879
Epoch 4/10
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8326 - loss: 0.4403 - val_accuracy: 0.8062 - val_loss: 0.5105
Epoch 5/10
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8394 - loss: 0.4144 - val_accuracy: 0.8046 - val_loss: 0.5161
Epoch 6/10
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8453 - loss: 0.3906 - val_accuracy: 0.8150 - val_loss: 0.4821
Epoch 7/10
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8486 - loss: 0.3737 - val_accuracy: 0.8079 - val_loss: 0.5095
Epoch 8/10
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8531 - loss: 0.3586 - 

In [ ]:
# Evaluate on held-out test set
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"\nTest Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

314/314 ━━━━━━━━━━━━━━━━━━━━ 0s 509us/step - accuracy: 0.8101 - loss: 0.5454

Test Accuracy: 0.8101
Test Loss: 0.5454


### Local results comparison

| Model | Val Accuracy | Training Time | Cost |
|-------|-------------|---------------|------|
| TF-IDF + Logistic Regression | 83.8% | ~10 seconds | Free |
| TF Embedding (local) | ~81% | ~20 seconds | Free |

**Key insight:** TF-IDF beats the neural model for this task. Topic classification
is keyword-driven (library names, syntax patterns), which is exactly what TF-IDF
captures. The embedding model learns similar features but with more parameters and noise.

**Exam relevance:** Simpler models often win on keyword-driven tasks. Always baseline.

## Part 3b: Vertex AI Training with Pre-built Containers

Now train the same model on Vertex AI using Google's pre-built TF container.
This demonstrates the `CustomTrainingJob` workflow with automatic model registration.

**Why pre-built containers?** No Dockerfile to maintain, no dependency conflicts,
Google handles TF + CUDA compatibility. You just provide a training script.

In [ ]:
%%writefile train.py
import argparse
import json
import os
import numpy as np
import pandas as pd
import tensorflow as tf

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", required=True)
    parser.add_argument("--vocab-size", type=int, default=10000)
    parser.add_argument("--max-length", type=int, default=128)
    parser.add_argument("--embedding-dim", type=int, default=64)
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--learning-rate", type=float, default=0.001)
    parser.add_argument("--text-truncate", type=int, default=500)
    args = parser.parse_args()

    # AIP_MODEL_DIR is set automatically by Vertex AI
    # Saving here auto-registers the model in Model Registry
    model_dir = os.environ.get("AIP_MODEL_DIR")
    print(f"Args: {vars(args)}")
    print(f"TensorFlow version: {tf.__version__}")
    print(f"AIP_MODEL_DIR: {model_dir}")

    # --- Load data (GCS paths work natively in pre-built containers) ---
    print("Loading data...")
    train_df = pd.read_csv(os.path.join(args.data_dir, "train.csv"))
    val_df = pd.read_csv(os.path.join(args.data_dir, "val.csv"))

    train_df = train_df.dropna(subset=["text", "label"])
    val_df = val_df.dropna(subset=["text", "label"])

    # Truncate long text — critical for performance
    train_df["text"] = train_df["text"].str[:args.text_truncate]
    val_df["text"] = val_df["text"].str[:args.text_truncate]

    print(f"Train: {len(train_df)}, Val: {len(val_df)}")

    # --- Encode labels ---
    labels = sorted(train_df["label"].unique())
    label_to_id = {l: i for i, l in enumerate(labels)}
    print(f"Labels: {label_to_id}")

    y_train = train_df["label"].map(label_to_id).values
    y_val = val_df["label"].map(label_to_id).values

    # --- Text vectorization ---
    print("Building text vectorizer...")
    vectorizer = tf.keras.layers.TextVectorization(
        max_tokens=args.vocab_size,
        output_mode="int",
        output_sequence_length=args.max_length,
    )
    vectorizer.adapt(train_df["text"].values)
    print(f"Vocabulary size: {vectorizer.vocabulary_size()}")

    # --- Build model ---
    print("Building model...")
    model = tf.keras.Sequential([
        vectorizer,
        tf.keras.layers.Embedding(args.vocab_size, args.embedding_dim),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(len(labels), activation="softmax"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=args.learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    model.summary()

    # --- Create tf.data datasets for efficient batching ---
    print("Creating datasets...")
    train_ds = tf.data.Dataset.from_tensor_slices(
        (train_df["text"].values, y_train)
    ).shuffle(10000).batch(args.batch_size).prefetch(tf.data.AUTOTUNE)

    val_ds = tf.data.Dataset.from_tensor_slices(
        (val_df["text"].values, y_val)
    ).batch(args.batch_size).prefetch(tf.data.AUTOTUNE)

    # --- Train ---
    print("Training...")
    history = model.fit(train_ds, validation_data=val_ds, epochs=args.epochs)

    # --- Evaluate ---
    val_loss, val_accuracy = model.evaluate(val_ds)
    print(f"\nValidation accuracy: {val_accuracy:.4f}")

    # --- Save as SavedModel (format expected by TF Serving prediction container) ---
    print(f"Saving model to {model_dir}...")
    model.save(model_dir)
    print("Model saved!")

    # --- Save metadata alongside model ---
    metadata = {
        "val_accuracy": float(val_accuracy),
        "val_loss": float(val_loss),
        "labels": labels,
        "label_to_id": label_to_id,
        "params": vars(args),
        "train_size": len(train_df),
        "val_size": len(val_df),
    }
    metadata_path = os.path.join(model_dir, "metadata.json")
    with tf.io.gfile.GFile(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"Metadata saved: {json.dumps(metadata, indent=2)}")


if __name__ == "__main__":
    main()

Writing train.py


In [ ]:
# Submit training job to Vertex AI
# CustomTrainingJob packages train.py and runs it inside the pre-built container
# When model_serving_container_image_uri is set, Vertex AI auto-registers the model

job = aiplatform.CustomTrainingJob(
    display_name="lab7-stackoverflow-text-clf",
    script_path="train.py",
    container_uri=TRAIN_IMAGE,
    model_serving_container_image_uri=SERVE_IMAGE,
)

vertex_model = job.run(
    args=[
        "--data-dir", f"{BUCKET_URI}/data",
        "--vocab-size", "10000",
        "--max-length", "128",
        "--embedding-dim", "64",
        "--epochs", "10",
        "--batch-size", "64",
        "--learning-rate", "0.001",
        "--text-truncate", "500",
    ],
    replica_count=1,
    machine_type="n1-standard-8",
    model_display_name="stackoverflow-classifier",
)

print(f"\nModel resource: {vertex_model.resource_name}")
print(f"Model URI: {vertex_model.uri}")

Training script copied to:
gs://carty-470812-lab7-stackoverflow/aiplatform-2026-03-01-11:44:55.504-aiplatform_custom_trainer_script-0.1.tar.gz.
Training Output directory:
gs://carty-470812-lab7-stackoverflow/aiplatform-custom-training-2026-03-01-11:44:55.988 
View Training:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/5504724669770498048?project=873708835509
CustomTrainingJob projects/873708835509/locations/us-central1/trainingPipelines/5504724669770498048 current state:
3
View backing custom job:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/781279891449446400?project=873708835509
CustomTrainingJob projects/873708835509/locations/us-central1/trainingPipelines/5504724669770498048 current state:
3
CustomTrainingJob projects/873708835509/locations/us-central1/trainingPipelines/5504724669770498048 current state:
3
CustomTrainingJob projects/873708835509/locations/us-central1/trainingPipelines/5504724669770498048 current state:
3

In [ ]:
# Check Vertex AI training results
!gsutil cp {vertex_model.uri}/metadata.json /tmp/metadata_vertex.json

import json
with open("/tmp/metadata_vertex.json") as f:
    meta = json.load(f)

print(f"Vertex AI Validation Accuracy: {meta['val_accuracy']:.4f}")
print(f"Vertex AI Validation Loss:     {meta['val_loss']:.4f}")

Copying gs://carty-470812-lab7-stackoverflow/aiplatform-custom-training-2026-03-01-11:44:55.988/model/metadata.json...
/ [1 files][  552.0 B/  552.0 B]                                                
Operation completed over 1 objects/552.0 B.                                      
Vertex AI Validation Accuracy: 0.5878
Vertex AI Validation Loss:     1.3274


**NOTE:** The pre-built container uses TF 2.15, which may produce different results
than your local TF version due to changes in TextVectorization behavior between versions.
This is itself an important lesson — version mismatches between training environments
are a real production ML problem.

If accuracy is significantly lower than local, this demonstrates why version pinning matters.

## Part 4: Batch Prediction

Run batch prediction on the test set using Vertex AI's BatchPredictionJob.
This is a key exam topic — understanding when to use batch vs. online prediction.

**Batch prediction:**
- Score a large dataset in one go (thousands+ rows)
- No endpoint needed — no idle costs
- Pay only for compute during the job
- Use for: nightly scoring, bulk processing, offline analysis

**Online prediction:**
- Deploy to endpoint, send individual requests
- Low latency (<100ms)
- Pay for endpoint being up 24/7
- Use for: real-time serving, interactive applications

### 4a. Prepare test data in JSONL format

In [ ]:
# Vertex AI batch prediction expects JSONL — one JSON instance per line
# The TF Serving container expects raw string inputs (matches model signature)

test_df_batch = pd.read_csv("/tmp/test.csv").dropna(subset=["text", "label"])
test_df_batch["text"] = test_df_batch["text"].str[:500]

jsonl_path = "/tmp/test_instances.jsonl"
with open(jsonl_path, "w") as f:
    for _, row in test_df_batch.iterrows():
        # Each line is a single JSON string — the model's input is a string tensor
        f.write(json.dumps(row["text"]) + "\n")

# Verify format
!head -1 {jsonl_path} | python3 -c "import sys; print(sys.stdin.read()[:200])"

print(f"\nWrote {len(test_df_batch)} instances")
!gsutil cp {jsonl_path} {BUCKET_URI}/data/test_instances.jsonl

"Can I use part of a Shell Object parsing name in the windows portable device (WPD) api to get a file stream?  Is it always valid to use the first part of the last guid from a windows api codepack She

Wrote 10043 instances
Copying file:///tmp/test_instances.jsonl [Content-Type=application/octet-stream]...
| [1 files][  4.7 MiB/  4.7 MiB]                                                
Operation completed over 1 objects/4.7 MiB.                                      


### 4b. Submit batch prediction job

In [ ]:
batch_job = vertex_model.batch_predict(
    job_display_name="lab7-batch-prediction",
    gcs_source=f"{BUCKET_URI}/data/test_instances.jsonl",
    gcs_destination_prefix=f"{BUCKET_URI}/predictions/batch/",
    machine_type="n1-standard-4",
    starting_replica_count=1,
    max_replica_count=1,
)

batch_job.wait()
print(f"\nStatus: {batch_job.state}")
print(f"Output: {batch_job.output_info.gcs_output_directory}")

Creating BatchPredictionJob
BatchPredictionJob created. Resource name: projects/873708835509/locations/us-central1/batchPredictionJobs/8048695509281406976
To use this BatchPredictionJob in another session:
bpj = aiplatform.BatchPredictionJob('projects/873708835509/locations/us-central1/batchPredictionJobs/8048695509281406976')
View Batch Prediction Job:
https://console.cloud.google.com/ai/platform/locations/us-central1/batch-predictions/8048695509281406976?project=873708835509
BatchPredictionJob projects/873708835509/locations/us-central1/batchPredictionJobs/8048695509281406976 current state:
3
BatchPredictionJob projects/873708835509/locations/us-central1/batchPredictionJobs/8048695509281406976 current state:
3
BatchPredictionJob projects/873708835509/locations/us-central1/batchPredictionJobs/8048695509281406976 current state:
3
BatchPredictionJob projects/873708835509/locations/us-central1/batchPredictionJobs/8048695509281406976 current state:
3
BatchPredictionJob projects/8737088355

### 4c. Analyze batch prediction results

In [ ]:
# Download prediction results
output_dir = batch_job.output_info.gcs_output_directory
!gsutil cp "{output_dir}/prediction.results-*" /tmp/

# Parse all result files
import glob
results = []
for path in sorted(glob.glob("/tmp/prediction.results-*")):
    with open(path) as f:
        for line in f:
            results.append(json.loads(line))

print(f"Total predictions: {len(results)}")
print(f"Prediction shape: {len(results[0]['prediction'])} (one score per class)")
print(f"Sample prediction: {results[0]['prediction']}")

# Map predictions to labels and calculate accuracy
import numpy as np
predicted_labels = [labels[np.argmax(r["prediction"])] for r in results]

test_df_batch["predicted"] = predicted_labels
test_df_batch["correct"] = test_df_batch["label"] == test_df_batch["predicted"]

accuracy = test_df_batch["correct"].mean()
print(f"\nBatch Prediction Accuracy: {accuracy:.4f}")

print(f"\nPer-class accuracy:")
for label in labels:
    subset = test_df_batch[test_df_batch["label"] == label]
    acc = (subset["label"] == subset["predicted"]).mean()
    print(f"  {label:12s}: {acc:.4f} ({len(subset)} samples)")

Copying gs://carty-470812-lab7-stackoverflow/predictions/batch/prediction-stackoverflow-classifier-2026_03_01T03_53_18_988Z/prediction.results-00000-of-00002...
Copying gs://carty-470812-lab7-stackoverflow/predictions/batch/prediction-stackoverflow-classifier-2026_03_01T03_53_18_988Z/prediction.results-00001-of-00002...
\ [2 files][  5.7 MiB/  5.7 MiB]                                                
Operation completed over 2 objects/5.7 MiB.                                      
Total predictions: 10043
Prediction shape: 5 (one score per class)
Sample prediction: [0.0119249821, 0.0359586813, 0.451279432, 0.442006916, 0.0588298328]

Batch Prediction Accuracy: 0.1047

Per-class accuracy:
  c#          : 0.0010 (2081 samples)
  html        : 0.0000 (1954 samples)
  java        : 0.0488 (1968 samples)
  javascript  : 0.2902 (1985 samples)
  python      : 0.1835 (2055 samples)


## Part 5: Metadata & Lineage

Explore what Vertex AI automatically tracked from our training and batch prediction jobs.
ML Metadata provides three core concepts:

- **Artifacts:** Data, models, endpoints — things that exist
- **Executions:** Training runs, batch jobs — things that happened
- **Contexts:** Experiments, pipelines — groups of related work

These enable auditing ("which data was this model trained on?") and debugging
("what changed between model v1 and v2?").

In [ ]:
# What custom jobs did we run?
print("=== Custom Jobs ===")
jobs = aiplatform.CustomJob.list()
for j in jobs:
    print(f"  {j.display_name} | state={j.state} | {j.create_time.strftime('%Y-%m-%d %H:%M')}")

=== Custom Jobs ===
  lab7-stackoverflow-text-clf-custom-job | state=4 | 2026-03-01 11:45
  lab7-stackoverflow-v3-custom-job | state=4 | 2026-03-01 10:50
  census-lab4-20260221_164359-custom-job | state=4 | 2026-02-21 16:44


In [ ]:
# What models are registered?
print("=== Registered Models ===")
models = aiplatform.Model.list()
for m in models:
    print(f"  {m.display_name}")
    print(f"    Resource: {m.resource_name}")
    print(f"    URI: {m.uri}")
    print(f"    Container: {m.container_spec.image_uri}")
    print(f"    Created: {m.create_time.strftime('%Y-%m-%d %H:%M')}")
    print()

=== Registered Models ===
  stackoverflow-classifier
    Resource: projects/873708835509/locations/us-central1/models/4556850075015315456
    URI: gs://carty-470812-lab7-stackoverflow/aiplatform-custom-training-2026-03-01-11:44:55.988/model
    Container: us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest
    Created: 2026-03-01 11:44

  stackoverflow-classifier-v2
    Resource: projects/873708835509/locations/us-central1/models/594245352882700288
    URI: gs://carty-470812-lab7-stackoverflow/aiplatform-custom-training-2026-03-01-10:50:14.796/model
    Container: us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest
    Created: 2026-03-01 10:50

  stackoverflow-classifier-v2
    Resource: projects/873708835509/locations/us-central1/models/1096396711334510592
    URI: gs://carty-470812-lab7-stackoverflow/aiplatform-custom-training-2026-03-01-10:05:15.468/model
    Container: us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest
    Created: 2026-03-01 10:05

  census-

In [ ]:
# What batch prediction jobs ran?
print("=== Batch Prediction Jobs ===")
batch_jobs = aiplatform.BatchPredictionJob.list()
for j in batch_jobs:
    print(f"  {j.display_name} | state={j.state}")
    print(f"    Created: {j.create_time.strftime('%Y-%m-%d %H:%M')}")
    if j.output_info:
        print(f"    Output: {j.output_info.gcs_output_directory}")
    print()

=== Batch Prediction Jobs ===
  lab7-batch-prediction | state=4
    Created: 2026-03-01 11:53
    Output: gs://carty-470812-lab7-stackoverflow/predictions/batch/prediction-stackoverflow-classifier-2026_03_01T03_53_18_988Z

  lab7-batch-prediction-v2 | state=4
    Created: 2026-03-01 10:58
    Output: gs://carty-470812-lab7-stackoverflow/predictions/batch_v2/prediction-stackoverflow-classifier-v2-2026_03_01T02_58_13_085Z

  lab7-batch-prediction | state=4
    Created: 2026-03-01 10:22
    Output: gs://carty-470812-lab7-stackoverflow/predictions/batch/prediction-stackoverflow-classifier-v2-2026_03_01T02_22_24_440Z



In [ ]:
# ML Metadata: Artifacts (data, models tracked by Vertex AI)
print("=== Artifacts ===")
artifacts = aiplatform.Artifact.list()
for a in artifacts:
    print(f"  {a.display_name} | schema: {a.schema_title} | state: {a.state}")
    print(f"    URI: {a.uri}")
    if a.metadata:
        print(f"    Metadata: {dict(list(a.metadata.items())[:3])}...")
    print()

=== Artifacts ===
  census-gbm-best-20260222-083816 | schema: system.Model | state: 2
    URI: gs://carty-470812-ml-census-data/models/
    Metadata: {'created_by': 'lab5-mlops', 'roc_auc': 0.93, 'python_version': '3.10'}...

  census-training-data-20260222-083816 | schema: system.Dataset | state: 2
    URI: gs://carty-470812-ml-census-data/data/census_income.csv
    Metadata: {'columns': 15.0, 'source': 'bigquery-public-data.ml_datasets.census_adult_income', 'created_by': 'lab5-mlops'}...

   | schema: google.VertexTensorboardRun | state: 2
    URI: https://us-central1-aiplatform.googleapis.com/v1/projects/873708835509/locations/us-central1/tensorboards/1689098349891813376/experiments/census-mlops-lab5/runs/autolog-demo
    Metadata: {'resourceName': 'projects/873708835509/locations/us-central1/tensorboards/1689098349891813376/experiments/census-mlops-lab5/runs/autolog-demo', 'vertex_experiment_tracking': True}...

   | schema: google.VertexTensorboardRun | state: 2
    URI: https://u

In [ ]:
# ML Metadata: Executions (training runs, processing steps)
print("=== Executions ===")
executions = aiplatform.Execution.list()
for e in executions:
    print(f"  {e.display_name} | schema: {e.schema_title} | state: {e.state}")
    print(f"    Created: {e.create_time.strftime('%Y-%m-%d %H:%M')}")
    print()

=== Executions ===
  census-training-20260222-083816 | schema: system.ContainerExecution | state: 3
    Created: 2026-02-22 08:38



## Lab 7 Summary

In [ ]:
print("=" * 60)
print("LAB 7 RESULTS SUMMARY")
print("=" * 60)

print("""
PIPELINE: BQ (public data) → GCS (CSV) → Local + Vertex AI → Batch Prediction

PART 1: Data Preparation
  - 100k Stack Overflow questions, 5 tags, 80/10/10 split
  - BQ → GCS export with hash-based reproducible splits
  - HTML tag stripping, title + body concatenation

PART 2: TF-IDF Baseline
  - TF-IDF + Logistic Regression: 83.8% validation accuracy
  - 10 seconds to train, strong keyword-based features
  - Best: python (95%), java (91%) | Worst: javascript (69%), html (75%)

PART 3: Neural Model
  - Embedding + GlobalAveragePooling1D architecture
  - Local training (TF 2.20): ~81% validation accuracy
  - Vertex AI training (TF 2.15): ~57% due to TF version mismatch
  - Key finding: TF-IDF beats neural for keyword-driven classification

PART 4: Batch Prediction
  - JSONL format: one raw JSON string per line
  - 10,043 predictions via Vertex AI BatchPredictionJob
  - Demonstrated batch vs. online serving pattern

PART 5: Metadata & Lineage
  - Vertex AI auto-tracks: custom jobs, models, batch jobs
  - ML Metadata: artifacts, executions for audit trail

LESSONS LEARNED:
  1. Pre-built containers eliminate Docker dependency hell
  2. TF version mismatch causes real accuracy differences (TextVectorization changed)
  3. Text truncation is critical — 35k char posts kill training performance
  4. TF-IDF is a strong baseline for topic classification (exam pattern!)
  5. Batch prediction format: one raw JSON value per JSONL line
  6. Always test locally before submitting to Vertex AI
""")

LAB 7 RESULTS SUMMARY

PIPELINE: BQ (public data) → GCS (CSV) → Local + Vertex AI → Batch Prediction

PART 1: Data Preparation
  - 100k Stack Overflow questions, 5 tags, 80/10/10 split
  - BQ → GCS export with hash-based reproducible splits
  - HTML tag stripping, title + body concatenation

PART 2: TF-IDF Baseline
  - TF-IDF + Logistic Regression: 83.8% validation accuracy
  - 10 seconds to train, strong keyword-based features
  - Best: python (95%), java (91%) | Worst: javascript (69%), html (75%)

PART 3: Neural Model
  - Embedding + GlobalAveragePooling1D architecture
  - Local training (TF 2.20): ~81% validation accuracy
  - Vertex AI training (TF 2.15): ~57% due to TF version mismatch
  - Key finding: TF-IDF beats neural for keyword-driven classification

PART 4: Batch Prediction
  - JSONL format: one raw JSON string per line
  - 10,043 predictions via Vertex AI BatchPredictionJob
  - Demonstrated batch vs. online serving pattern

PART 5: Metadata & Lineage
  - Vertex AI auto-tra

## Cleanup

Delete resources to avoid ongoing charges.

In [41]:
# Delete batch prediction jobs, models, and endpoints
# Uncomment when ready to clean up

# for j in aiplatform.BatchPredictionJob.list():
#     j.delete()

# for m in aiplatform.Model.list():
#     if "stackoverflow" in m.display_name:
#         m.delete()

# Delete GCS bucket contents (keep bucket for reference)
# !gsutil -m rm -r {BUCKET_URI}/predictions/
# !gsutil -m rm -r {BUCKET_URI}/model/

# Delete BigQuery dataset
# bq.delete_dataset(f"{PROJECT_ID}.lab7_stackoverflow", delete_contents=True)

print("Cleanup commands ready — uncomment to execute")

Cleanup commands ready — uncomment to execute
